# 4. Predict — Score New Student Records
Load the best trained model and predict dropout risk for new students. Returns: dropout probability, at-risk flag, and risk level (Low/Medium/High).

In [ ]:
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path('outputs')

model      = joblib.load(OUTPUT_DIR / 'best_model.pkl')
model_name = joblib.load(OUTPUT_DIR / 'best_model_name.pkl')
feat_names = joblib.load(OUTPUT_DIR / 'feature_names.pkl')

print(f"Loaded model: {model_name}")
print(f"Expected features ({len(feat_names)}): {feat_names[:8]} ...")


## 4.1 Feature builder
Converts a student record dict into the feature vector expected by the model.

In [ ]:
def build_feature_row(student: dict, feat_names: list) -> pd.DataFrame:
    """
    Convert student dict to model feature row.

    Expected keys:
      pell, firstgen        : 'Y', 'N', or 'Unknown'
      gender                : 'Female' or 'Male'
      term_type             : 'Spring' or 'Fall'
      regstat               : 1-4  (1=First-time, 2=Transfer, 3=Readmit, 4=Continuing)
      accumgpa              : float (cumulative GPA)
      creditenr             : int (credits enrolled this term)
      terms_enrolled_so_far : int (how many terms in this episode so far)
      gpa_change            : float (GPA change from prior term; 0 for first term)
      credit_change          : float (credit change from prior term; 0 for first term)
      gpa_trend_3term        : float (GPA vs. rolling avg of prior 2 terms; 0 for first term)
      credit_trend_3term     : float (credits vs. rolling avg of prior 2 terms; 0 for first term)
      ever_stopped_out       : int (1 if episode had a Stop-out/Readmit in a PRIOR term, else 0)
      citizen               : 1-4
      ethnicmultirace       : int (1-27)
      school                : 'AD', 'CC', 'EN', 'SL', 'SM'
      u_g                   : 'U', 'G', 'D'
      degtype               : 'BA', 'BS', 'MS', 'MBA', 'PHD', 'CRT', etc.
    """
    row = {
        'pell_encoded':          {'Y': 1.0, 'N': 0.0, 'Unknown': 0.5}.get(student.get('pell', 'Unknown'), 0.5),
        'firstgen_encoded':      {'Y': 1.0, 'N': 0.0, 'Unknown': 0.5}.get(student.get('firstgen', 'Unknown'), 0.5),
        'gender_encoded':        1 if student.get('gender') == 'Female' else 0,
        'term_season_encoded':   1 if student.get('term_type') == 'Spring' else 0,
        'regstat_encoded':       float(student.get('regstat', 4)),
        'accumgpa':              float(student.get('accumgpa', 0.0)),
        'creditenr':             float(student.get('creditenr', 12)),
        'terms_enrolled_so_far': float(student.get('terms_enrolled_so_far', 1)),
        'gpa_change':            float(student.get('gpa_change', 0.0)),
        'credit_change':         float(student.get('credit_change', 0.0)),
        'gpa_trend_3term':       float(student.get('gpa_trend_3term', 0.0)),
        'credit_trend_3term':    float(student.get('credit_trend_3term', 0.0)),
        'ever_stopped_out':      float(student.get('ever_stopped_out', 0)),
        'citizen_encoded':       float(student.get('citizen', 4)),
        'ethnicity_encoded':     float(student.get('ethnicmultirace', 0)),
    }
    # One-hot: school
    for s in ['AD', 'CC', 'EN', 'SL', 'SM']:
        row[f'school_{s}'] = 1 if student.get('school') == s else 0
    # One-hot: u_g
    for ug in ['D', 'G', 'U']:
        row[f'ug_{ug}'] = 1 if student.get('u_g') == ug else 0
    # One-hot: degtype (any column in feat_names starting with deg_)
    for col in [f for f in feat_names if f.startswith('deg_')]:
        row[col] = 1 if student.get('degtype') == col.replace('deg_', '') else 0

    df = pd.DataFrame([row])
    for col in feat_names:
        if col not in df.columns:
            df[col] = 0
    return df[feat_names]


def predict_student(student: dict, threshold: float = None) -> dict:
    # threshold=None -> use optimal threshold from training
    if threshold is None:
        try:
            threshold = joblib.load(OUTPUT_DIR / 'optimal_threshold.pkl')
        except FileNotFoundError:
            threshold = 0.3  # fallback if not yet computed
    X = build_feature_row(student, feat_names)
    prob = model.predict_proba(X)[0, 1]
    pred = int(prob >= threshold)
    risk = 'High' if prob >= 0.70 else 'Medium' if prob >= 0.40 else 'Low'
    return {
        'model':               model_name,
        'dropout_probability': round(float(prob), 4),
        'at_risk':             pred,
        'risk_level':          risk,
    }

print("predict_student() ready.")


## 4.2 Score prediction_output.xlsx and save final_predicted_output.xlsx


In [ ]:
# Load optimal threshold determined during evaluation
try:
    optimal_threshold = joblib.load(OUTPUT_DIR / 'optimal_threshold.pkl')
    print(f'Using optimal threshold: {optimal_threshold:.4f}')
except FileNotFoundError:
    optimal_threshold = 0.3
    print(f'optimal_threshold.pkl not found, using fallback: {optimal_threshold}')

# Load student records to score
input_df = pd.read_excel('prediction_output.xlsx')
print(f'Loaded {len(input_df):,} records from prediction_output.xlsx')

# Exclude graduated students -- not at-risk candidates (matches training methodology)
scored_df = input_df[input_df['academic_state'] != 'Graduated'].copy()
print(f'Scoring {len(scored_df):,} records (excluded {len(input_df) - len(scored_df):,} Graduated)')

# Episode-derived features aren't available in a single-term snapshot, so
# default to the same 'first term of episode' fallback used for new/unseen
# students: terms_enrolled_so_far=1, all momentum/trend/history features=0
dropout_probs, at_risk_flags = [], []
for _, row in scored_df.iterrows():
    student = row.to_dict()
    student.setdefault('terms_enrolled_so_far', 1)
    student.setdefault('gpa_change', 0.0)
    student.setdefault('credit_change', 0.0)
    student.setdefault('gpa_trend_3term', 0.0)
    student.setdefault('credit_trend_3term', 0.0)
    student.setdefault('ever_stopped_out', 0)
    result = predict_student(student, threshold=optimal_threshold)
    dropout_probs.append(result['dropout_probability'])
    at_risk_flags.append('YES' if result['at_risk'] else 'NO')

scored_df['dropout_probability'] = dropout_probs
scored_df['risk_flag'] = at_risk_flags

# Recombine with excluded Graduated rows (left blank) to preserve the full record set
graduated_df = input_df[input_df['academic_state'] == 'Graduated'].copy()
final_df = pd.concat([scored_df, graduated_df], axis=0).sort_index()

output_path = OUTPUT_DIR / 'final_predicted_output.xlsx'
final_df.to_excel(output_path, index=False)
print(f'Saved {len(final_df):,} records to {output_path}')

final_df[['student_id', 'academic_state', 'dropout_probability', 'risk_flag']].head(10)


## 4.3 Batch prediction on a DataFrame

In [ ]:
# Example: score multiple students at once
batch = pd.DataFrame([s for _, s in students])
batch['label'] = [l for l, _ in students]

results = [predict_student(row.to_dict()) for _, row in batch.iterrows()]
batch['dropout_probability'] = [r['dropout_probability'] for r in results]
batch['risk_level']          = [r['risk_level']          for r in results]

batch[['label', 'pell', 'accumgpa', 'regstat', 'dropout_probability', 'risk_level']]
